# Base

In [86]:
import pandas as pd
import numpy as np
import datetime as dt

# ===============================================
import sys
from pathlib import Path

project_root = Path().resolve().parents[0]
sys.path.append(str(project_root))
# ===============================================


from pipeline.coleta import get_pmc_index, get_pmc_pesos
from sktime.transformations.hierarchical.aggregate import Aggregator

import warnings
warnings.filterwarnings('ignore')


# Módulos skitme


In [62]:

# ==========
#  Pipeline
# ==========
from sktime.forecasting.compose import TransformedTargetForecaster, ForecastingPipeline

# =============================
#  Modelos univariados básicos
# =============================
from sktime.forecasting.statsforecast import (
                                                StatsForecastAutoARIMA, StatsForecastAutoETS, 
                                                StatsForecastAutoCES, StatsForecastAutoTBATS
                                            )

# ====================
#  Modelos compostos
# ====================
from sktime.forecasting.compose import AutoEnsembleForecaster

# ===============
#  Regressors
# ===============
from lightgbm import LGBMRegressor, DaskLGBMRegressor
from catboost import CatBoostRegressor

# ============================
#  Métodos de reconciliação
# ============================
from sktime.forecasting.reconcile import ( 
                                            BottomUpReconciler, TopdownReconciler, 
                                            OptimalReconciler, ReconcilerForecaster
                                        )
# ===============
#  Transformers
# ===============
from sktime.transformations.series.boxcox import LogTransformer, BoxCoxTransformer
from sktime.transformations.series.detrend import Detrender, Deseasonalizer, ConditionalDeseasonalizer
from sktime.transformations.series.difference import Differencer
from sktime.transformations.compose import OptionalPassthrough

# ==================
#  Cross Validation
# ==================
from sktime.split import TemporalTrainTestSplitter, ExpandingWindowSplitter, SlidingWindowSplitter
from sktime.forecasting.model_evaluation import evaluate
from sktime.forecasting.model_selection import ForecastingOptunaSearchCV, ForecastingRandomizedSearchCV, ForecastingGridSearchCV
from optuna.distributions import CategoricalDistribution, FloatDistribution, IntDistribution

# ===========
#  Métricas
# ===========
from sktime.performance_metrics.forecasting import (
                                                    MeanAbsoluteError, MeanAbsoluteScaledError,
                                                    MeanAbsolutePercentageError, MeanSquaredError
                                                    )

# Coleta

In [21]:
from sktime.split import temporal_train_test_split

In [49]:
pmc_raw = get_pmc_index('restrita_sem_aberturas').dropna()
pmc_raw

pesos_raw = get_pmc_pesos('restrita_sem_aberturas')
pesos_raw


pmc_agg = pmc_raw \
    .reset_index() \
    .merge(pesos_raw, on='Atividades', how='left') \
    .assign(indice_pond = lambda df: df['nindice'] * df['Pesos']/100) \
    .groupby(['Atividades', 'Data'])[['indice_pond']].last() \
    .pipe( Aggregator().fit_transform )

test_size=24
train, test = temporal_train_test_split(y=pmc_agg, test_size = test_size)
fh =  range(1, test_size+1)

# Fit

In [14]:
arima = StatsForecastAutoARIMA(sp=12)
tbats = StatsForecastAutoTBATS(seasonal_periods=12)
ets =  StatsForecastAutoETS(season_length=12)
ces = StatsForecastAutoCES(season_length=12)
lgbm = LGBMRegressor(verbosity=-1)
ensemble = AutoEnsembleForecaster(forecasters=[arima, tbats, ets, ces], regressor=lgbm)


In [119]:
pipe = TransformedTargetForecaster(steps=[
    # ('log', LogTransformer()),
    ('deseason', Deseasonalizer(sp=12)),
    ('diff', Differencer()),
    ('forecaster', ensemble),
    ('reconciler', TopdownReconciler())

])

In [120]:
pipe.fit(train, fh=fh)
preds = pipe.predict()

In [121]:
# preds

In [122]:
# agg_preds = Aggregator().fit_transform(preds)
# agg_preds

In [123]:
metrics(test, preds, train).round(4)

,MeanAbsoluteError,MeanAbsolutePercentageError,MeanSquaredError,MeanAbsoluteScaledError
1. Combustíveis e lubrificantes,0.4380,3.56,0.4846,0.5871
"2. Hipermercados, supermercados, produtos alimentícios, bebidas e fumo",4.2331,7.20,5.9337,2.2448
"3. Tecidos, vestuário e calçados",1.4330,22.77,2.5312,2.8161
4. Móveis e eletrodomésticos,0.8438,10.48,1.0778,1.3645
"5. Artigos farmacêuticos, médicos, ortopédicos, de perfumaria e cosméticos",0.5880,5.38,0.6670,1.4418
"6. Livros, jornais, revistas e papelaria",0.1199,40.55,0.1402,1.5856
"7. Equipamentos e materiais para escritório, informática e comunicação",0.1463,8.81,0.2390,0.8378
8. Outros artigos de uso pessoal e doméstico,1.2597,12.73,1.6354,1.7665
__total,7.4015,6.83,10.9532,1.9849


In [112]:
mae_overall = MeanAbsoluteError()
mae_raw = MeanAbsoluteError(multilevel='raw_values')

mape_overall = MeanAbsolutePercentageError(symmetric=False)
mape_raw = MeanAbsolutePercentageError(symmetric=False, multilevel='raw_values')

# mae_raw(test, agg_preds)
mape_raw(y_true=test, y_pred=agg_preds).multiply(100).round(2)

,MeanAbsolutePercentageError
1. Combustíveis e lubrificantes,3.69
"2. Hipermercados, supermercados, produtos alimentícios, bebidas e fumo",9.00
"3. Tecidos, vestuário e calçados",21.18
4. Móveis e eletrodomésticos,13.47
"5. Artigos farmacêuticos, médicos, ortopédicos, de perfumaria e cosméticos",6.29
"6. Livros, jornais, revistas e papelaria",19.07
"7. Equipamentos e materiais para escritório, informática e comunicação",10.85
8. Outros artigos de uso pessoal e doméstico,15.46
__total,8.06


In [70]:
def metrics(y_true, y_pred, y_train):
    mae_overall = MeanAbsoluteError()
    mae_raw = MeanAbsoluteError(multilevel='raw_values')

    mape_overall = MeanAbsolutePercentageError(symmetric=False)
    mape_raw = MeanAbsolutePercentageError(symmetric=False, multilevel='raw_values')

    rmse_raw = MeanSquaredError(square_root=True, multilevel='raw_values')

    mase_raw = MeanAbsoluteScaledError(sp=12, multilevel='raw_values')

    mae = mae_raw(y_true=y_true, y_pred=y_pred)
    mape = mape_raw(y_true=y_true, y_pred=y_pred).multiply(100).round(2)
    rmse = rmse_raw(y_true=y_true, y_pred=y_pred)
    mase = mase_raw(y_true=y_true, y_pred=y_pred, y_train=y_train)

    metrics_df = pd.concat([mae, mape, rmse, mase], axis=1)

    return metrics_df


In [50]:
# pmc_raw['Total'] = 'Total'

pmc_agg = (
    pmc_raw
    .reset_index()
    .merge(pesos_raw, on='Atividades')
    .assign(
        Total = 'Total',
        indice_pond = lambda df: df['nindice'] * df['Pesos']/100

    )
    .set_index(['Total','Atividades', 'Data'])
    [['indice_pond']]
    .pipe(Aggregator().fit_transform)

)